<a href="https://colab.research.google.com/github/02-056-Vebri/mahadata-spark-load-forecasting/blob/main/Vebri_Yanti_123140056_Tugas_Mahadata_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
!pip install -q pyspark==3.5.5
!java -version

[0.002s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.002s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
openjdk version "21.0.12.1" 2026-08-18
OpenJDK Runtime Environment (build 21.0.12.1+1-1-24.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12.1+1-1-24.04.4-Ubuntu, mixed mode, sharing)


In [13]:
from pyspark.sql import SparkSession
import pyspark, sys

spark = (SparkSession.builder
         .master("local[*]")
         .appName("smoke-test")
         .config("spark.driver.memory", "4g")
         .getOrCreate())

print("Python :", sys.version.split()[0])
print("PySpark:", pyspark.__version__)
print("Spark  :", spark.version)

df = spark.range(1_000_000).selectExpr("id", "id % 10 as grp")
df.groupBy("grp").count().show()

Python : 3.13.16
PySpark: 3.5.5
Spark  : 3.5.5
+---+------+
|grp| count|
+---+------+
|  0|100000|
|  7|100000|
|  6|100000|
|  9|100000|
|  5|100000|
|  1|100000|
|  3|100000|
|  8|100000|
|  2|100000|
|  4|100000|
+---+------+



In [14]:
!pip install -q pyspark==3.5.5 kagglehub

import os, kagglehub
from pyspark.sql import SparkSession

# 1. Buat sesi Spark
spark = (SparkSession.builder
         .master("local[*]")
         .appName("eksplorasi-mahadata")
         .config("spark.driver.memory", "4g")
         .getOrCreate())
print("Spark:", spark.version)

# 2. Unduh dataset dari Kaggle
path = kagglehub.dataset_download("jeanmidev/smart-meters-in-london")
print("Path dataset:", path)

# 3. Cari alamat file yang dibutuhkan
target = ["block_0.csv", "weather_hourly_darksky.csv", "acorn_details.csv",
          "informations_households.csv", "uk_bank_holidays.csv"]
lokasi = {}
for root, dirs, files in os.walk(path):
    for f in files:
        if f in target:
            lokasi[f] = os.path.join(root, f)
for k, v in lokasi.items():
    print(k, "->", v)

# 4. Muat Block 0
df = spark.read.csv(lokasi["block_0.csv"], header=True, inferSchema=True)
df.printSchema()
df.show(5, truncate=False)
print("Jumlah baris:", df.count())

Spark: 3.5.5
Using Colab cache for faster access to the 'smart-meters-in-london' dataset.
Path dataset: /kaggle/input/smart-meters-in-london
weather_hourly_darksky.csv -> /kaggle/input/smart-meters-in-london/weather_hourly_darksky.csv
acorn_details.csv -> /kaggle/input/smart-meters-in-london/acorn_details.csv
uk_bank_holidays.csv -> /kaggle/input/smart-meters-in-london/uk_bank_holidays.csv
informations_households.csv -> /kaggle/input/smart-meters-in-london/informations_households.csv
block_0.csv -> /kaggle/input/smart-meters-in-london/daily_dataset/daily_dataset/block_0.csv
root
 |-- LCLid: string (nullable = true)
 |-- day: date (nullable = true)
 |-- energy_median: double (nullable = true)
 |-- energy_mean: double (nullable = true)
 |-- energy_max: double (nullable = true)
 |-- energy_count: integer (nullable = true)
 |-- energy_std: double (nullable = true)
 |-- energy_sum: double (nullable = true)
 |-- energy_min: double (nullable = true)

+---------+----------+-------------------+

In [15]:
import os

hh = None
for root, dirs, files in os.walk(path):
    if "halfhourly" in root and "block_0.csv" in files:
        hh = os.path.join(root, "block_0.csv")
print("Alamat half-hourly:", hh)

df_hh = spark.read.csv(hh, header=True, inferSchema=True)
df_hh.printSchema()
df_hh.show(5, truncate=False)
print("Jumlah baris:", df_hh.count())
print("Jumlah rumah:", df_hh.select("LCLid").distinct().count())

Alamat half-hourly: /kaggle/input/smart-meters-in-london/halfhourly_dataset/halfhourly_dataset/block_0.csv
root
 |-- LCLid: string (nullable = true)
 |-- tstp: timestamp (nullable = true)
 |-- energy(kWh/hh): string (nullable = true)

+---------+-------------------+--------------+
|LCLid    |tstp               |energy(kWh/hh)|
+---------+-------------------+--------------+
|MAC000002|2012-10-12 00:30:00| 0            |
|MAC000002|2012-10-12 01:00:00| 0            |
|MAC000002|2012-10-12 01:30:00| 0            |
|MAC000002|2012-10-12 02:00:00| 0            |
|MAC000002|2012-10-12 02:30:00| 0            |
+---------+-------------------+--------------+
only showing top 5 rows

Jumlah baris: 1222670
Jumlah rumah: 50


In [16]:
from pyspark.sql import functions as F

# 1. Rapikan nama kolom, ubah energi ke angka
raw = (df_hh
       .withColumnRenamed("energy(kWh/hh)", "energy_raw")
       .withColumn("energy", F.col("energy_raw").cast("double")))

# 2. Cek kualitas data
total = raw.count()
bad = raw.filter(F.col("energy").isNull()).count()
print(f"Total baris        : {total}")
print(f"Energi tidak valid : {bad} ({bad/total:.4%})")
print("Contoh nilai bermasalah:")
raw.filter(F.col("energy").isNull()).select("energy_raw").distinct().show(5)

# 3. Buang baris tidak valid
clean = raw.filter(F.col("energy").isNotNull() & (F.col("energy") >= 0))
print("Baris setelah dibersihkan:", clean.count())

# 4. Agregasi per timestamp: total energi dan jumlah rumah tercatat
agg = (clean.groupBy("tstp")
            .agg(F.sum("energy").alias("energy_sum"),
                 F.count("LCLid").alias("n_houses"))
            .orderBy("tstp"))

# 5. Fitur waktu
agg = (agg
       .withColumn("hour", F.hour("tstp"))
       .withColumn("minute", F.minute("tstp"))
       .withColumn("dow", F.dayofweek("tstp"))   # 1=Minggu ... 7=Sabtu
       .withColumn("is_weekend", F.when(F.col("dow").isin(1, 7), 1).otherwise(0)))

agg.cache()
print("Jumlah titik waktu:", agg.count())
agg.show(5)
agg.select("tstp", "energy_sum", "n_houses").describe().show()

# 6. Intip struktur file cuaca
weather = spark.read.csv(lokasi["weather_hourly_darksky.csv"], header=True, inferSchema=True)
weather.printSchema()
weather.show(3, truncate=False)

Total baris        : 1222670
Energi tidak valid : 50 (0.0041%)
Contoh nilai bermasalah:
+----------+
|energy_raw|
+----------+
|      Null|
+----------+

Baris setelah dibersihkan: 1222620
Jumlah titik waktu: 39247
+-------------------+----------+--------+----+------+---+----------+
|               tstp|energy_sum|n_houses|hour|minute|dow|is_weekend|
+-------------------+----------+--------+----+------+---+----------+
|2011-12-03 09:00:00|     0.149|       1|   9|     0|  7|         1|
|2011-12-03 09:30:00|     0.154|       1|   9|    30|  7|         1|
|2011-12-03 10:00:00|     0.768|       1|  10|     0|  7|         1|
|2011-12-03 10:30:00|     1.179|       1|  10|    30|  7|         1|
|2011-12-03 11:00:00|     0.588|       1|  11|     0|  7|         1|
+-------------------+----------+--------+----+------+---+----------+
only showing top 5 rows

+-------+------------------+-----------------+
|summary|        energy_sum|         n_houses|
+-------+------------------+-----------------

In [17]:
# 1. Sebaran jumlah rumah per bulan
agg.createOrReplaceTempView("agg")
spark.sql("""
    SELECT date_format(tstp, 'yyyy-MM') AS bulan,
           COUNT(*) AS titik_waktu,
           MIN(n_houses) AS min_rumah,
           ROUND(AVG(n_houses), 1) AS rata_rumah,
           MAX(n_houses) AS max_rumah
    FROM agg GROUP BY 1 ORDER BY 1
""").show(40, truncate=False)

# 2. Cek duplikasi waktu di data cuaca
dup = weather.groupBy("time").count().filter("count > 1").count()
print("Timestamp cuaca duplikat:", dup)

# 3. Gabungkan cuaca (dibulatkan ke jam)
w = (weather.select(
        F.date_trunc("hour", "time").alias("w_hour"),
        "temperature", "apparentTemperature", "dewPoint",
        "humidity", "windSpeed", "pressure", "visibility"))

data = (agg.withColumn("w_hour", F.date_trunc("hour", "tstp"))
           .join(w, on="w_hour", how="left")
           .withColumn("energy_per_house", F.col("energy_sum") / F.col("n_houses")))
data.cache()

print("Baris tanpa cuaca:",
      data.filter(F.col("apparentTemperature").isNull()).count())

# 4. Korelasi dengan konsumsi
for c in ["temperature", "apparentTemperature", "dewPoint",
          "humidity", "windSpeed", "pressure", "visibility"]:
    r1 = data.stat.corr("energy_sum", c)
    r2 = data.stat.corr("energy_per_house", c)
    print(f"{c:20s} total: {r1:+.3f} | per rumah: {r2:+.3f}")

+-------+-----------+---------+----------+---------+
|bulan  |titik_waktu|min_rumah|rata_rumah|max_rumah|
+-------+-----------+---------+----------+---------+
|2011-12|1374       |1        |1.0       |1        |
|2012-01|1488       |1        |1.0       |1        |
|2012-02|1392       |1        |1.0       |1        |
|2012-03|1488       |1        |2.2       |4        |
|2012-04|1440       |4        |4.5       |5        |
|2012-05|1488       |5        |6.6       |9        |
|2012-06|1440       |7        |8.8       |9        |
|2012-07|1488       |9        |9.8       |10       |
|2012-08|1488       |9        |10.0      |10       |
|2012-09|1440       |9        |12.0      |18       |
|2012-10|1488       |18       |35.8      |49       |
|2012-11|1440       |47       |48.9      |49       |
|2012-12|1488       |42       |48.7      |50       |
|2013-01|1488       |46       |48.0      |48       |
|2013-02|1344       |40       |48.0      |48       |
|2013-03|1488       |47       |48.0      |48  

In [18]:
# 1. Potong data: periode dengan jumlah rumah stabil, buang baris tanpa cuaca
d = data.filter((F.col("tstp") >= "2012-11-01") &
                F.col("apparentTemperature").isNotNull())
d.cache()
print("Titik waktu setelah dipotong:", d.count())
print("Rentang n_houses:", d.agg(F.min("n_houses"), F.max("n_houses")).first())

# 2. Korelasi ulang, per 30 menit
print("\nKorelasi per 30 menit (data terpotong):")
for c in ["temperature", "apparentTemperature", "dewPoint",
          "humidity", "windSpeed", "pressure", "visibility"]:
    print(f"{c:20s} total: {d.stat.corr('energy_sum', c):+.3f}")

# 3. Korelasi pada rata-rata harian
daily = (d.groupBy(F.to_date("tstp").alias("day"))
           .agg(F.avg("energy_per_house").alias("e_rumah"),
                F.avg("apparentTemperature").alias("app_temp"),
                F.avg("temperature").alias("temp")))
print("\nKorelasi rata-rata harian:")
print("apparentTemperature:", round(daily.stat.corr("e_rumah", "app_temp"), 3))
print("temperature        :", round(daily.stat.corr("e_rumah", "temp"), 3))

Titik waktu setelah dipotong: 23229
Rentang n_houses: Row(min(n_houses)=39, max(n_houses)=50)

Korelasi per 30 menit (data terpotong):
temperature          total: -0.262
apparentTemperature  total: -0.277
dewPoint             total: -0.289
humidity             total: -0.015
windSpeed            total: +0.118
pressure             total: -0.015
visibility           total: +0.010

Korelasi rata-rata harian:
apparentTemperature: -0.849
temperature        : -0.846


In [19]:
# 1. Pembersihan sesuai artikel: buang null DAN nol
n_nol = raw.filter(F.col("energy") == 0).count()
clean2 = raw.filter(F.col("energy").isNotNull() & (F.col("energy") > 0))
print("Baris bernilai nol     :", n_nol)
print("Baris setelah dibersihkan:", clean2.count())

# 2. Agregasi ulang
agg2 = (clean2.groupBy("tstp")
        .agg(F.sum("energy").alias("energy_sum"),
             F.count("LCLid").alias("n_houses"))
        .orderBy("tstp")
        .withColumn("hour", F.hour("tstp"))
        .withColumn("dow", F.dayofweek("tstp"))
        .withColumn("w_hour", F.date_trunc("hour", "tstp"))
        .withColumn("tgl", F.to_date("tstp")))

# 3. Gabung cuaca
data2 = (agg2.join(w, on="w_hour", how="left")
             .filter(F.col("apparentTemperature").isNotNull()))
data2.cache()
print("Titik waktu data penuh:", data2.count())
data2.agg(F.min("n_houses"), F.max("n_houses"),
          F.round(F.avg("n_houses"), 1)).show()

# 4. Struktur file hari libur
holidays = spark.read.csv(lokasi["uk_bank_holidays.csv"], header=True, inferSchema=True)
holidays.printSchema()
holidays.show(5, truncate=False)

# 5. Korelasi: data penuh vs terpotong
terpotong = data2.filter(F.col("tstp") >= "2012-11-01")
for nama, d in [("PENUH", data2), ("TERPOTONG", terpotong)]:
    print(f"\n{nama}  (n={d.count()})")
    for c in ["temperature", "apparentTemperature"]:
        print(f"  {c:20s} {d.stat.corr('energy_sum', c):+.3f}")

Baris bernilai nol     : 1652
Baris setelah dibersihkan: 1220968
Titik waktu data penuh: 39243
+-------------+-------------+-----------------------+
|min(n_houses)|max(n_houses)|round(avg(n_houses), 1)|
+-------------+-------------+-----------------------+
|            1|           50|                   31.1|
+-------------+-------------+-----------------------+

root
 |-- Bank holidays: date (nullable = true)
 |-- Type: string (nullable = true)

+-------------+--------------------------------------------+
|Bank holidays|Type                                        |
+-------------+--------------------------------------------+
|2012-12-26   |Boxing Day                                  |
|2012-12-25   |Christmas Day                               |
|2012-08-27   |Summer bank holiday                         |
|2012-05-06   |Queen?s Diamond Jubilee (extra bank holiday)|
|2012-04-06   |Spring bank holiday (substitute day)        |
+-------------+--------------------------------------------+


In [11]:
import time, pandas as pd
from pyspark.ml import Pipeline
from pyspark.ml.feature import OneHotEncoder, VectorAssembler, PolynomialExpansion, StandardScaler
from pyspark.ml.regression import LinearRegression, DecisionTreeRegressor
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator

# 1. Fitur akhir pekan + hari libur
hol = (holidays.select(F.col("Bank holidays").alias("tgl"))
       .withColumn("is_hol", F.lit(1)).distinct())
print("Rentang hari libur:", hol.agg(F.min("tgl"), F.max("tgl")).first())

base = (data2.join(hol, on="tgl", how="left")
        .withColumn("is_weekend",
                    F.when(F.col("dow").isin(1, 7) | (F.col("is_hol") == 1), 1).otherwise(0))
        .withColumn("hour_d", F.col("hour").cast("double"))
        .withColumn("label", F.col("energy_sum")))
base.cache()
print("Hari libur/akhir pekan:", base.filter("is_weekend=1").count(), "titik waktu")

# 2. Fungsi evaluasi dan eksperimen
def metrik(pred):
    return {m: RegressionEvaluator(labelCol="label", predictionCol="prediction",
                                   metricName=m).evaluate(pred)
            for m in ["r2", "mae", "mse", "rmse"]}

def jalankan(df, nama, suhu="apparentTemperature"):
    train, test = df.randomSplit([0.8, 0.2], seed=42)
    train.cache(); test.cache()
    print(f"\n=== {nama}: latih={train.count()}, uji={test.count()} ===")

    ohe = OneHotEncoder(inputCols=["hour_d"], outputCols=["hour_ohe"])
    asm = VectorAssembler(inputCols=["n_houses", suhu, "is_weekend", "hour_ohe"],
                          outputCol="raw")
    poly = PolynomialExpansion(degree=2, inputCol="raw", outputCol="poly")
    sc = StandardScaler(inputCol="poly", outputCol="features",
                        withMean=False, withStd=True)
    ev_cv = RegressionEvaluator(labelCol="label", metricName="r2")
    lambdas = [1e-4, 1e-3, 1e-2]
    hasil = []

    def catat(model_name, model, t_fit, tuning, cv_r2):
        t0 = time.time()
        pred = model.transform(test); pred.count()
        t_test = time.time() - t0
        r = metrik(pred)
        hasil.append({"data": nama, "model": model_name, "tuning": tuning,
                      "cv_R2": cv_r2, "R2": r["r2"], "MAE": r["mae"],
                      "MSE": r["mse"], "RMSE": r["rmse"],
                      "fit_s": round(t_fit, 1), "test_s": round(t_test, 2)})
        print("selesai:", model_name)

    def linier(reg, en):
        return LinearRegression(featuresCol="features", labelCol="label",
                                regParam=reg, elasticNetParam=en, maxIter=100)

    # Plain (tanpa regularisasi, tanpa tuning)
    pipe = Pipeline(stages=[ohe, asm, poly, sc, linier(0.0, 0.0)])
    t0 = time.time(); m = pipe.fit(train)
    catat("Plain", m, time.time() - t0, "NA", None)

    # Ridge, Lasso, Elastic net (tuning dengan cross-validation)
    for nama_m, en in [("Ridge", 0.0), ("Lasso", 1.0), ("ElasticNet", None)]:
        est = linier(0.0, 0.5 if en is None else en)
        pipe = Pipeline(stages=[ohe, asm, poly, sc, est])
        gb = ParamGridBuilder().addGrid(est.regParam, lambdas)
        if en is None:
            gb = gb.addGrid(est.elasticNetParam, [0.0, 0.5, 1.0])
        cv = CrossValidator(estimator=pipe, estimatorParamMaps=gb.build(),
                            evaluator=ev_cv, numFolds=3, seed=42, parallelism=2)
        t0 = time.time(); cvm = cv.fit(train); t_fit = time.time() - t0
        best = cvm.bestModel.stages[-1]
        tuning = f"lambda={best.getRegParam()}, alpha={best.getElasticNetParam()}"
        catat(nama_m, cvm, t_fit, tuning, max(cvm.avgMetrics))

    # Decision tree (kedalaman 5, tanpa polinomial/penskalaan)
    asm_t = VectorAssembler(inputCols=["n_houses", suhu, "is_weekend", "hour_d"],
                            outputCol="features")
    dtr = DecisionTreeRegressor(featuresCol="features", labelCol="label",
                                maxDepth=5, seed=42)
    t0 = time.time(); m = Pipeline(stages=[asm_t, dtr]).fit(train)
    catat("DTR", m, time.time() - t0, "maxDepth=5", None)

    return pd.DataFrame(hasil)

# 3. Jalankan: data penuh (replikasi) dan terpotong (variasi)
h_penuh = jalankan(base, "PENUH")
h_pot = jalankan(base.filter(F.col("tstp") >= "2012-11-01"), "TERPOTONG")

hasil_semua = pd.concat([h_penuh, h_pot], ignore_index=True)
hasil_semua.round(3).to_csv("hasil_model.csv", index=False)
print(hasil_semua.round(3).to_string())

Rentang hari libur: Row(min(tgl)=datetime.date(2012, 2, 1), max(tgl)=datetime.date(2014, 12, 26))
Hari libur/akhir pekan: 12030 titik waktu

=== PENUH: latih=31306, uji=7937 ===
selesai: Plain
selesai: Ridge
selesai: Lasso
selesai: ElasticNet
selesai: DTR

=== TERPOTONG: latih=18496, uji=4733 ===
selesai: Plain
selesai: Ridge
selesai: Lasso
selesai: ElasticNet
selesai: DTR
        data       model                    tuning  cv_R2     R2    MAE     MSE   RMSE  fit_s  test_s
0      PENUH       Plain                        NA    NaN  0.943  1.753   6.262  2.502   18.4    2.08
1      PENUH       Ridge  lambda=0.0001, alpha=0.0  0.942  0.943  1.754   6.262  2.502  200.9    1.90
2      PENUH       Lasso   lambda=0.001, alpha=1.0  0.941  0.886  2.996  12.652  3.557  177.3    1.76
3      PENUH  ElasticNet  lambda=0.0001, alpha=0.0  0.942  0.943  1.754   6.262  2.502  454.5    0.99
4      PENUH         DTR                maxDepth=5    NaN  0.918  1.939   9.126  3.021   19.2    1.22
5  TERPOTONG

In [21]:
hasil_semua.round(3).to_csv("hasil_semua.csv", index=False)